# JAX Memory Diagnostics

This notebook is for finding TPU/JAX memory bottlenecks in the Hyperattn runner without using the dataset path. It uses synthetic token batches, the same model config shape as `jax_runner_notebook.ipynb`, and opt-in compile/runtime probes.

The default cells are safe: they estimate tensor sizes and build the model. The heavy probes are disabled until you set the `RUN_*` flags to `True`.

## 1. Imports and Config

In [ ]:
import math
import time
from dataclasses import asdict
from functools import partial

import jax
import jax.numpy as jnp
import numpy as np
import torch

import jax_model
from jax_model.layers import block_forward
from jax_model.ops import layer_norm, linear_cross_entropy
from jax_model.rope import precompute_rope_cache
from model import EfficientHGConfig, EfficientHypergraphLM

print("jax backend:", jax.default_backend())
print("jax devices:", jax.device_count(), jax.devices())

In [ ]:
seed = 1337

# Mirror the runner defaults. Change these first when bisecting.
batch_size = 16
block_size = 2048
vocab_size = 50257
n_embd = 384
n_head = 6
span_widths = (2, 4, 8, 16, 32, 64)
local_window = 256
compression_block = 64
block_layout = tuple(["attn"] + (["span"] * 4 + ["hca"] + ["span"] * 4) * 3 + ["span"] * 4 + ["hca"] + ["span"] * 2)

# Probe knobs.
attention_backend = "chunked"          # keep this chunked for large T
span_backend = "fused"                 # avoids [B,T,C*len(widths)] span concat tensors
remat_blocks = True                    # trades compute for lower backward activation memory
scan_span_runs = True                  # compiles repeated span runs as lax.scan loops
use_bfloat16 = jax.default_backend() == "tpu"
compute_dtype = jnp.bfloat16 if use_bfloat16 else jnp.float32

# Heavy probes are off by default. Turn on one at a time.
RUN_COMPILE_PROBES = False
RUN_RUNTIME_SMOKE = False
RUN_HLO_SCAN = False

assert batch_size % jax.device_count() == 0
print("batch_size:", batch_size, "block_size:", block_size, "compute_dtype:", compute_dtype)
print("blocks:", len(block_layout), "layout:", block_layout)

## 2. Static Memory Estimates

These are not XLA peak-memory numbers. They are quick estimates for the largest obvious tensors, useful for deciding what to probe next.

In [ ]:
def gib(n):
    return n / 1024**3


def dtype_bytes(dtype):
    return np.dtype(jnp.dtype(dtype)).itemsize


def fmt_bytes(n):
    if n >= 1024**3:
        return f"{gib(n):.2f} GiB"
    if n >= 1024**2:
        return f"{n / 1024**2:.1f} MiB"
    return f"{n / 1024:.1f} KiB"


B, T, C, H, V = batch_size, block_size, n_embd, n_head, vocab_size
D = C // H
db = dtype_bytes(compute_dtype)
fp32 = dtype_bytes(jnp.float32)
n_blocks = len(block_layout)
n_spans = sum(1 for x in block_layout if x == "span")
n_hca = sum(1 for x in block_layout if x in ("hca", "mem"))
n_attn = sum(1 for x in block_layout if x == "attn")
span_runs = []
run = 0
for kind in block_layout:
    if kind == "span":
        run += 1
    elif run:
        span_runs.append(run)
        run = 0
if run:
    span_runs.append(run)
max_span_run = max(span_runs) if span_runs else 0

full_logits = B * T * V * fp32
hidden = B * T * C * db
span_materialized = B * T * C * len(span_widths) * db
mlp_hidden = B * T * 4 * C * db
largest_scanned_span_residual_stack = max_span_run * hidden
largest_scanned_span_mlp_stack = max_span_run * mlp_hidden
manual_attn_scores = B * H * T * T * fp32
n_chunks = math.ceil(T / local_window)
chunked_attn_scores = B * n_chunks * H * local_window * (2 * local_window) * fp32
hca_scores = B * H * T * (math.ceil(T / compression_block) + 1) * fp32
saved_residuals_floor = n_blocks * hidden

rows = [
    ("full logits [B,T,V]", full_logits, "avoid in train/eval loss"),
    ("one residual stream [B,T,C]", hidden, "saved many times in backward"),
    ("saved residuals floor", saved_residuals_floor, "only a lower bound"),
    ("one materialized span concat", span_materialized, "per span block before projection"),
    ("one MLP expansion", mlp_hidden, "per block before proj"),
    ("largest scanned span residual stack", largest_scanned_span_residual_stack, "no-remat scan backward can save this"),
    ("largest scanned span MLP stack", largest_scanned_span_mlp_stack, "matches TPU [run,B,T,4C] OOM shapes"),
    ("manual attention scores", manual_attn_scores, "do not use at long T"),
    ("chunked attention scores", chunked_attn_scores, "expected local attention scale"),
    ("one HCA score tensor", hca_scores, "compressed-memory attention"),
]

print(f"attn={n_attn} span={n_spans} hca={n_hca} total_blocks={n_blocks} span_runs={span_runs}")
for name, n, note in rows:
    print(f"{name:34s} {fmt_bytes(n):>12s}   {note}")

## 3. Build Params and Estimate Persistent State

In [ ]:
torch.manual_seed(seed)
torch_cfg = EfficientHGConfig(
    vocab_size=vocab_size,
    block_size=block_size,
    n_embd=n_embd,
    n_head=n_head,
    span_widths=span_widths,
    local_window=local_window,
    compression_block=compression_block,
    dropout=0.0,
    block_layout=block_layout,
)
params, cfg = jax_model.from_torch_model(EfficientHypergraphLM(torch_cfg))


def tree_nbytes(tree):
    total = 0
    for x in jax.tree.leaves(tree):
        if hasattr(x, "size") and hasattr(x, "dtype"):
            total += int(x.size) * np.dtype(x.dtype).itemsize
    return total


param_bytes = tree_nbytes(params)
print("config:", asdict(cfg))
print(f"parameters: {jax_model.count_parameters(params) / 1e6:.2f}M")
print("fp32 params:", fmt_bytes(param_bytes))
print("AdamW s1+s2:", fmt_bytes(2 * param_bytes))
print("params + AdamW state:", fmt_bytes(3 * param_bytes))
print("rough params + AdamW + bf16 compute copy:", fmt_bytes(3 * param_bytes + param_bytes // 2))

## 4. Mesh, Sharding, and Synthetic Batch

In [ ]:
n_devices = jax.device_count()
if n_devices > 1:
    from jax.sharding import Mesh, NamedSharding, PartitionSpec

    mesh = Mesh(np.asarray(jax.devices()), ("data",))
    replicated = NamedSharding(mesh, PartitionSpec())
    data_sharded = NamedSharding(mesh, PartitionSpec("data"))

    def replicate(tree):
        return jax.device_put(tree, replicated)

    def shard_batch(a):
        return jax.device_put(jnp.asarray(a), data_sharded)
else:
    def replicate(tree):
        return tree

    shard_batch = jnp.asarray

params = replicate(params)

key = jax.random.PRNGKey(seed)
x_key, y_key = jax.random.split(key)
xb = jax.random.randint(x_key, (batch_size, block_size), 0, vocab_size, dtype=jnp.int32)
yb = jax.random.randint(y_key, (batch_size, block_size), 0, vocab_size, dtype=jnp.int32)
xb, yb = shard_batch(xb), shard_batch(yb)
print("xb:", xb.shape, "yb:", yb.shape)

## 5. Diagnostic Loss Variants

In [ ]:
def cast_for_compute(p):
    if compute_dtype == jnp.float32:
        return p
    return jax.tree.map(
        lambda a: a.astype(compute_dtype) if hasattr(a, "dtype") and jnp.issubdtype(a.dtype, jnp.floating) else a,
        p,
    )


def loss_diag(p, x, y, *, attention_backend, span_backend, remat_blocks, scan_span_runs):
    p = cast_for_compute(p)
    return jax_model.loss(
        p,
        x,
        y,
        cfg,
        attention_backend=attention_backend,
        span_backend=span_backend,
        remat_blocks=remat_blocks,
        scan_span_runs=scan_span_runs,
    )


def make_loss(attention_backend, span_backend, remat_blocks, scan_span_runs):
    return lambda p, x, y: loss_diag(
        p,
        x,
        y,
        attention_backend=attention_backend,
        span_backend=span_backend,
        remat_blocks=remat_blocks,
        scan_span_runs=scan_span_runs,
    )

## 6. Compile-Time Memory Probes

Turn `RUN_COMPILE_PROBES = True` above to run these. If a config fails at compile time, the printed config is the useful signal. Start with the fused/remat rows.

In [ ]:
def summarize_compiled(compiled):
    memory_analysis = getattr(compiled, "memory_analysis", None)
    if memory_analysis is not None:
        try:
            print("memory_analysis:", memory_analysis())
        except Exception as e:
            print("memory_analysis unavailable:", repr(e))

    try:
        cost = compiled.cost_analysis()
        if isinstance(cost, list):
            cost = cost[0] if cost else {}
        for k in sorted(cost):
            v = cost[k]
            if "bytes" in k.lower() or "flops" in k.lower() or "optimal" in k.lower():
                print(f"cost {k}: {v}")
    except Exception as e:
        print("cost_analysis unavailable:", repr(e))


def compile_probe(name, attention_backend, span_backend, remat_blocks, scan_span_runs):
    print("\n===", name, "===")
    print("attention_backend:", attention_backend, "span_backend:", span_backend, "remat_blocks:", remat_blocks, "scan_span_runs:", scan_span_runs)
    loss_fn = make_loss(attention_backend, span_backend, remat_blocks, scan_span_runs)
    grad_fn = jax.jit(jax.value_and_grad(loss_fn))
    t0 = time.time()
    lowered = grad_fn.lower(params, xb, yb)
    print(f"lowered in {time.time() - t0:.1f}s")
    t0 = time.time()
    compiled = lowered.compile()
    print(f"compiled in {time.time() - t0:.1f}s")
    summarize_compiled(compiled)
    return compiled


probe_configs = [
    ("chunked + fused + remat + scanned spans", "chunked", "fused", True, True),
    ("chunked + materialized + remat + scanned spans", "chunked", "materialized", True, True),
    ("chunked + fused + remat + unrolled spans", "chunked", "fused", True, False),
    # This is expected to be memory-hungry: scan backward saves [run,B,T,*] activations.
    ("chunked + fused + no remat + scanned spans", "chunked", "fused", False, True),
]

compiled_probe = None
if RUN_COMPILE_PROBES:
    for cfg_name, attn, span, remat, scan in probe_configs:
        try:
            compiled_probe = compile_probe(cfg_name, attn, span, remat, scan)
        except Exception as e:
            print("FAILED:", cfg_name, repr(e))
            break
else:
    print("Compile probes disabled. Set RUN_COMPILE_PROBES = True in the config cell.")

## 7. Runtime Smoke Probe

Turn `RUN_RUNTIME_SMOKE = True` after a compile probe succeeds. This runs one value-and-grad step without optimizer state, so it isolates model backward memory from optimizer memory.

In [ ]:
if RUN_RUNTIME_SMOKE:
    loss_fn = make_loss(attention_backend, span_backend, remat_blocks, scan_span_runs)
    grad_fn = jax.jit(jax.value_and_grad(loss_fn))
    t0 = time.time()
    loss_value, grads = grad_fn(params, xb, yb)
    loss_value.block_until_ready()
    print(f"value_and_grad completed in {time.time() - t0:.1f}s loss={float(loss_value):.4f}")
    print("grad bytes:", fmt_bytes(tree_nbytes(grads)))
else:
    print("Runtime smoke disabled. Set RUN_RUNTIME_SMOKE = True in the config cell.")

## 8. HLO Text Scan

This is useful when a backend silently chooses a bad lowering. It does not prove peak memory, but it can show whether a full-logits shaped op is still present.

In [ ]:
if RUN_HLO_SCAN:
    loss_fn = make_loss(attention_backend, span_backend, remat_blocks, scan_span_runs)
    lowered = jax.jit(jax.value_and_grad(loss_fn)).lower(params, xb, yb)
    text = lowered.as_text()
    needles = [
        f"{batch_size}x{block_size}x{vocab_size}",
        str((batch_size, block_size, vocab_size)),
        "50257",
        "dot_general",
        "while",
        "remat",
    ]
    for needle in needles:
        print(f"{needle!r}:", text.count(needle))
    print("HLO text chars:", len(text))
else:
    print("HLO scan disabled. Set RUN_HLO_SCAN = True in the config cell.")

## Reading the Results

- If `chunked + fused + remat + scanned spans` compiles but unrolled spans do not, compile graph size was the bottleneck.
- If scanned fused spans compile but materialized spans do not, the span concat is the next bottleneck.
- If no-remat scanned spans fails with shapes like `[run, B, T, 4C]`, that is expected: scan backward is saving per-layer MLP/residual activations.
- If fused no-remat fails but fused remat compiles, saved activations across the deep stack are the bottleneck.
- If value-and-grad works here but the runner train step crashes, optimizer state/update memory is the bottleneck.
- If HLO contains a full `[batch, seq, vocab]` shaped value, something is still calling `jax_model.forward()` in a train/eval path.
- Keep `attention_backend="chunked"` for long sequences. `manual` attention is intentionally not probed here because it materializes `[B,H,T,T]`.